In [38]:
!pip install pytest requests pytest-mock -q

In [26]:
%%writefile test_biblioteca_smoke.py

import pytest
import requests

BASE = 'http://localhost:8000/api'

@pytest.mark.smoke
class TestBibliotecaSmoke:

    def test_01_api_disponible(self):
        try:
            response = requests.get(f"{BASE}/health", timeout=3)
            assert response.status_code == 200
            assert response.json()["status"] == "ok"
        except requests.exceptions.ConnectionError:
            pytest.skip("Servidor no disponible en este entorno")

    def test_02_login_funciona(self):
        try:
            payload = {"username": "estudiante_test", "password": "pass123"}
            response = requests.post(f"{BASE}/auth/login", json=payload, timeout=3)
            assert response.status_code == 200
            assert "access_token" in response.json()
        except requests.exceptions.ConnectionError:
            pytest.skip("Servidor no disponible en este entorno")

    def test_03_listar_libros_responde(self):
        try:
            response = requests.get(f"{BASE}/libros", timeout=3)
            assert response.status_code == 200
            assert isinstance(response.json(), list)
        except requests.exceptions.ConnectionError:
            pytest.skip("Servidor no disponible en este entorno")
base_de_datos_conectada(self):
        try:
    def test_04_
            response = requests.get(f"{BASE}/health", timeout=3)
            assert response.json()["db"] == "connected"
        except requests.exceptions.ConnectionError:
            pytest.skip("Servidor no disponible en este entorno")

    def test_05_modulo_multas_responde(self):
        try:
            response = requests.get(f"{BASE}/multas/estudiante_test", timeout=3)
            assert response.status_code in [200, 404]
            assert response.status_code != 500
        except requests.exceptions.ConnectionError:
            pytest.skip("Servidor no disponible en este entorno")

Overwriting test_biblioteca_smoke.py


In [32]:
import os
os.makedirs("biblioteca", exist_ok=True)

In [37]:
%%writefile biblioteca/__init__.py
# módulo biblioteca

Writing biblioteca/__init__.py


In [33]:
%%writefile biblioteca/prestamos.py

from datetime import datetime

_prestamos = {}  # libro_id -> {estudiante_id, fecha, devuelto}

def registrar_prestamo(libro_id, estudiante_id, fecha=None):
    _prestamos[libro_id] = {
        "estudiante_id": estudiante_id,
        "fecha": fecha or datetime.now(),
        "devuelto": False
    }

def registrar_devolucion(libro_id, estudiante_id):
    if libro_id in _prestamos:
        _prestamos[libro_id]["devuelto"] = True

def calcular_disponibilidad(libro_id):
    prestamo = _prestamos.get(libro_id)
    if not prestamo:
        return True
    return prestamo["devuelto"]

def obtener_fecha_prestamo(libro_id):
    return _prestamos.get(libro_id, {}).get("fecha")

Writing biblioteca/prestamos.py


In [34]:
%%writefile biblioteca/multas.py

TARIFA_DIARIA = 500

def calcular_multa(dias_retraso):
    return dias_retraso * TARIFA_DIARIA

Writing biblioteca/multas.py


In [35]:
%%writefile test_prestamos_regression.py

import pytest
from datetime import datetime, timedelta
from unittest.mock import patch
from biblioteca.prestamos import (
    calcular_disponibilidad,
    registrar_prestamo,
    registrar_devolucion,
)
from biblioteca.multas import calcular_multa

TARIFA_DIARIA = 500

@pytest.mark.regression
class TestPrestamosRegression:

    def setup_method(self):
        # Limpiar estado entre tests
        from biblioteca import prestamos
        prestamos._prestamos.clear()

    def test_libro_prestado_no_esta_disponible(self):
        registrar_prestamo("LIB-001", "EST-001")
        assert calcular_disponibilidad("LIB-001") is False

    def test_libro_disponible_tras_devolucion(self):
        registrar_prestamo("LIB-002", "EST-002")
        registrar_devolucion("LIB-002", "EST-002")
        assert calcular_disponibilidad("LIB-002") is True

    def test_bug_2024_031_no_disponible_con_prestamo_antiguo(self):
        fecha_antigua = datetime.now() - timedelta(days=45)
        with patch("biblioteca.prestamos.obtener_fecha_prestamo", return_value=fecha_antigua):
            registrar_prestamo("LIB-003", "EST-003", fecha=fecha_antigua)
            assert calcular_disponibilidad("LIB-003") is False, (
                "BUG-2024-031: préstamo de +30 días no debe marcarse como disponible"
            )

    def test_multa_no_afecta_disponibilidad_de_otro_libro(self):
        registrar_prestamo("LIB-005", "EST-004")  # genera multa
        assert calcular_disponibilidad("LIB-004") is True  # LIB-004 nunca fue prestado

    def test_calculo_multa_tres_dias_retraso(self):
        assert calcular_multa(dias_retraso=3) == 1500

Overwriting test_prestamos_regression.py


In [36]:
!pytest test_biblioteca_smoke.py test_prestamos_regression.py -v --tb=short

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: mock-3.15.1, typeguard-4.5.1, anyio-4.13.0, langsmith-0.7.30
collected 10 items                                                             

test_biblioteca_smoke.py::TestBibliotecaSmoke::test_01_api_disponible SKIPPED [ 10%]
test_biblioteca_smoke.py::TestBibliotecaSmoke::test_02_login_funciona SKIPPED [ 20%]
test_biblioteca_smoke.py::TestBibliotecaSmoke::test_03_listar_libros_responde SKIPPED [ 30%]
test_biblioteca_smoke.py::TestBibliotecaSmoke::test_04_base_de_datos_conectada SKIPPED [ 40%]
test_biblioteca_smoke.py::TestBibliotecaSmoke::test_05_modulo_multas_responde SKIPPED [ 50%]
test_prestamos_regression.py::TestPrestamosRegression::test_libro_prestado_no_esta_disponible PASSED [ 60%]
test_prestamos_regression.py::TestPrestamosRegression::test_libro_disponible_tras_devol